# DA3METRIC-LARGE · Vehicle-Anchored Scale/Shift Correction + Full Evaluation  *(v6 — fixed)*

**Combined pipeline** unifying:
- `predict_intrinsics_da3.py` — per-sequence DA3 camera intrinsic prediction  
- `shift_scale_correction_v2.py` — vehicle size priors, weighted anchor logic  
- `DA_3_metric_analysis_premasked.ipynb` — DA3 inference, KITTI indexing, plots

---

## Pipeline
| Pass | What happens |
|------|-------------|
| **Pass 1** | Load pre-generated vehicle masks → run DA3 depth → predict per-sequence intrinsics → extract vehicle bbox anchors → RANSAC+WLS → global correction |
| **Pass 2** | Apply global correction to raw DA3 depth → evaluate all 5 scale variants vs LiDAR GT → write per-mask CSV |
| **Analysis** | 13 analysis plots + terminal summary table |

## Key Idea
The **vehicle-anchored** variant uses **no LiDAR ground truth** at test time.  
It estimates depth scale purely from:
- Predicted camera intrinsics (DA3 temporal joint method)
- Known vehicle physical dimensions (height ≈ 1.5 m, width ≈ 1.82 m)
- Bounding boxes of pre-segmented vehicle masks (SAM3 output from `inst_seg_notebook.ipynb`)

## Correction model (new switch `ONLY_SCALE`)
| `ONLY_SCALE` | Model fitted | Shift |
|---|---|---|
| `False` *(default)* | `depth = scale · raw + shift` | fitted |
| `True` | `depth = scale · raw` | fixed at **0.00** |

## Vehicle Size Priors  *(keys now match the mask `sub_cat` names)*
| `sub_cat` | Height | Width | Reliability |
|---|---|---|---|
| `personal_vehicle_headlights` | 1.50 m ± 0.12 | 1.82 m ± 0.12 | 0.90 |
| `personal_vehicle_taillights` | 1.50 m ± 0.12 | 1.82 m ± 0.12 | 0.92 |

## What changed vs v5
1. **Root-cause fix** — `VEHICLE_PRIORS` was keyed `vehicle_headlights/taillights`, but the masks are named `personal_vehicle_*`, so *zero* anchors were ever built and the fit silently fell back to scale 1 / shift 0 (hence VA ≡ Unscaled).
2. **Fail loudly** — too few anchors now raises an error instead of returning an identity fit; Pass 1 reports every vehicle `sub_cat` that has no prior.
3. **Mask diagnostic cell** — prints bbox/aspect statistics and saves an overlay montage so you can verify the masks really cover the *whole* vehicle.
4. **Focal-length policy** (`INTRIN_FOCAL_POLICY`) — DA3 predicts `fx ≈ 1.8 × fy` on every drive; pixels are square, so the default now enforces `fx = fy = fy_pred`.
5. **`ONLY_SCALE` switch** (see above) with automatic re-fit from `vehicle_anchors.csv` when you flip it (no need to rerun DA3).
6. **Anchor sanity report** after the fit (anchor-vs-raw ratios, height/width agreement, per-sub_cat counts).
7. **Bug fixes** — WLS now weights with √w and no longer builds an N×N diagonal matrix; scale bounds in RANSAC; results CSV is overwritten instead of appended (v5 double-counted every mask); `main()` / `__main__` block removed from the plotting cell (it re-ran the whole pipeline inside that cell); Pass 2 anchor diagnostics now use the *same* code path as Pass 1.


## Cell 0 · GPU Selection & Imports

In [ ]:
# ── Optional: select GPU ──────────────────────────────────────────────────────
import os; os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# ── Standard library ──────────────────────────────────────────────────────────
import csv, glob, json, logging, math, os, sys, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import torchvision.transforms as T

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# ── Depth Anything 3 ──────────────────────────────────────────────────────────
# Install: git clone https://github.com/ByteDance-Seed/Depth-Anything-3
#          cd Depth-Anything-3 && pip install -e .
from depth_anything_3.api import DepthAnything3

print("Imports OK  |  CUDA available:", torch.cuda.is_available())

## Cell 1 · Logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("da3_va")
logging.getLogger("transformers").setLevel(logging.ERROR)

## Cell 2 · Configuration  *(edit paths here)*

All user-facing settings are collected here.

* `ONLY_SCALE` — `False` (default): fit **scale + shift**. `True`: fit **scale only**, shift is fixed at 0.00.
* `INTRIN_FOCAL_POLICY` — how predicted `fx`/`fy` are used (`square_fy` default, see Cell 4).
* `SKIP_PASS1 = True` re-uses the cached correction. If the cache was produced with a different `ONLY_SCALE` / focal policy it is automatically **re-fitted from `vehicle_anchors.csv`** (cheap — no DA3 inference).
* `SKIP_INFERENCE = True` jumps straight to analysis from an existing CSV.
* `OVERWRITE_RESULTS = True` starts a fresh results CSV each Pass 2 run (v5 appended, which duplicated every row).


In [ ]:
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR      = "/home/arka/Downloads/MMDE/cache"

# ── Dataset ───────────────────────────────────────────────────────────────────
KITTI_ROOT     = "/home/arka/Downloads/MMDE/datasets/KITTI/complete"
TEST_ROOT      = os.path.join(KITTI_ROOT, "test")
TRAIN_ROOT     = os.path.join(KITTI_ROOT, "train")
USE_TRAIN      = False

# ── Pre-generated mask store (output of inst_seg_notebook.ipynb) ──────────────
#    Layout (already inside the 'test' split folder):
#      MASK_ROOT_TEST|TRAIN / <date> / <drive> / image_02/data / <frame_stem> /
MASK_ROOT_TEST  = "/home/arka/Downloads/MMDE/outputs/inst_seg_output/test"
MASK_ROOT_TRAIN = "/home/arka/Downloads/MMDE/outputs/inst_seg_output/train"

# ── Depth clipping ────────────────────────────────────────────────────────────
MIN_DEPTH      = 1e-3
MAX_DEPTH      = 80.0

# ── KITTI default focal length (for DA3 metric conversion formula) ────────────
FOCAL_LENGTH_X = 721.5577

# ── Mask confidence gate ──────────────────────────────────────────────────────
MIN_MASK_SCORE = 0.50

# ── DA3 model identifiers ─────────────────────────────────────────────────────
DEPTH_MODEL_ID   = "depth-anything/DA3METRIC-LARGE"
INTRIN_MODEL_ID  = "depth-anything/DA3NESTED-GIANT-LARGE"

# ── Intrinsic prediction parameters ──────────────────────────────────────────
INTRIN_CHUNK_SIZE  = 20
INTRIN_PROCESS_RES = 504

# ── RANSAC parameters ─────────────────────────────────────────────────────────
RANSAC_ITERATIONS  = 1000
RANSAC_INLIER_THR  = 2.5    # metres — loosened to improve inlier rate on full-vehicle masks
MIN_ANCHORS_FOR_FIT = 4
MIN_PIXEL_DIM       = 8

# ── Correction model ──────────────────────────────────────────────────────────
#   ONLY_SCALE = False (default) →  depth = scale * raw + shift   (scale AND shift fitted)
#   ONLY_SCALE = True            →  depth = scale * raw           (shift fixed at 0.00)
ONLY_SCALE   = True
SCALE_BOUNDS = (0.2, 5.0)     # a fitted scale outside this range is rejected as implausible

# ── Intrinsics policy ─────────────────────────────────────────────────────────
# DA3 predicts fx ≈ 1.8 × fy on every KITTI drive (pixels are square, so fx≈fy in reality).
#   "as_predicted" : use DA3 fx and fy untouched (v5 behaviour)
#   "square_fy"    : fx = fy = fy_pred   (default – fy is the one close to KITTI's ~707-721 px)
#   "square_fx"    : fx = fy = fx_pred
#   "square_mean"  : fx = fy = (fx_pred + fy_pred) / 2
INTRIN_FOCAL_POLICY  = "as_predicted"
REUSE_INTRINSICS_CSV = True   # True → skip DA3 intrinsic prediction for drives already in INTRINSICS_CSV
assert INTRIN_FOCAL_POLICY in ("as_predicted", "square_fy", "square_fx", "square_mean")

# ── Mask diagnostic (verifies the masks really cover the whole vehicle) ───────
RUN_MASK_DIAGNOSTIC = True
DIAG_N_FRAMES       = 60
DIAG_N_OVERLAYS     = 12

# ── Output ────────────────────────────────────────────────────────────────────
MAIN_OUTPUT_DIR = "/home/arka/Downloads/MMDE/outputs/da3_vehicle_anchored_output/only_scale/as_predicted"
OUTPUT_CSV_PATH = os.path.join(MAIN_OUTPUT_DIR, "DA3_VA_results.csv")
ANCHORS_CSV     = os.path.join(MAIN_OUTPUT_DIR, "vehicle_anchors.csv")
INTRINSICS_CSV  = os.path.join(MAIN_OUTPUT_DIR, "sequence_intrinsics.csv")
PLOT_DIR        = os.path.join(MAIN_OUTPUT_DIR, "plots")
SCALE_CACHE     = os.path.join(MAIN_OUTPUT_DIR, "global_scale_shift.json")
SAVE_NPY        = False             # if you want to save the .npy files 

# ── Run control ───────────────────────────────────────────────────────────────
START_INDEX     = 0
SKIP_PASS1      = False   # True → reuse SCALE_CACHE from a previous run
SKIP_INFERENCE  = False   # True → skip Pass 2, go straight to analysis
SKIP_PLOTS      = False
OVERWRITE_RESULTS = True   # True → Pass 2 starts a fresh results CSV (False → append / resume)

os.makedirs(MAIN_OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
log.info("Configuration loaded. DEVICE=%s", DEVICE)
print("Configuration loaded.")
print(f"  ONLY_SCALE = {ONLY_SCALE}  ->  "
      f"{'scale only (shift fixed at 0.00)' if ONLY_SCALE else 'scale + shift'}")
print(f"  INTRIN_FOCAL_POLICY = {INTRIN_FOCAL_POLICY}")

## Cell 3 · Vehicle Size Priors

Physical dimensions derived from `shift_scale_correction_v2.py → EXTENDED_PRIORS`.

**The dictionary keys must equal the `sub_cat` strings stored in the mask store** — in v5 they did not
(`vehicle_headlights` vs the real `personal_vehicle_headlights`), which produced 0 anchors.

| `sub_cat` (mask store) | Real height | height_std | Real width | width_std | Reliability |
|---|---|---|---|---|---|
| `personal_vehicle_headlights` | 1.50 m | 0.12 | 1.82 m | 0.12 | 0.90 |
| `personal_vehicle_taillights` | 1.50 m | 0.12 | 1.82 m | 0.12 | 0.92 |

> **Assumption to verify:** these priors describe the **whole vehicle** bounding box. The diagnostic cell
> after Cell 8 prints bbox statistics and saves an overlay montage so you can confirm the masks cover the
> whole car and not just the lamps.
>
> **Note on "length" dimension:** left as a future extension; height and width already give two
> independent depth anchors per instance.


In [ ]:
@dataclass
class VehiclePrior:
    sub_cat:            str
    real_height:        Optional[float]
    real_width:         Optional[float]
    height_std:         float = 0.20
    width_std:          float = 0.20
    anchor_reliability: float = 1.0

    @property
    def height_weight(self) -> float:
        return (self.anchor_reliability / self.height_std ** 2
                if self.real_height else 0.0)

    @property
    def width_weight(self) -> float:
        return (self.anchor_reliability / self.width_std ** 2
                if self.real_width else 0.0)


# Priors are for the FULL VEHICLE bounding box (mask covers whole car).
# KEYS MUST MATCH the `sub_cat` strings produced by the mask store
# (see the mask diagnostic cell, which lists every vehicle sub_cat it finds).
# personal_vehicle_headlights = front-facing vehicle; personal_vehicle_taillights = rear-facing.
# Height is always meaningful (top of roof to road-level).
# Width is only meaningful when the vehicle is roughly frontal/rear-on
# (aspect-ratio check in extract_vehicle_anchors disables width for side-on views).
VEHICLE_PRIORS: Dict[str, VehiclePrior] = {
    "personal_vehicle_headlights": VehiclePrior(
        sub_cat="personal_vehicle_headlights",
        real_height=1.50, height_std=0.12,   # cars: 1.38–1.62 m (sedan/SUV range)
        real_width=1.82,  width_std=0.12,    # cars: 1.70–1.94 m
        anchor_reliability=0.90,
    ),
    "personal_vehicle_taillights": VehiclePrior(
        sub_cat="personal_vehicle_taillights",
        real_height=1.50, height_std=0.12,
        real_width=1.82,  width_std=0.12,
        anchor_reliability=0.92,
    ),
}
print("Vehicle priors:", list(VEHICLE_PRIORS.keys()))


## Cell 4 · Camera Intrinsics  *(from `predict_intrinsics_da3.py`)*

DA3NESTED-GIANT-LARGE is used to predict per-sequence camera intrinsics via
temporal joint estimation. Intrinsics are predicted **once per drive** and
cached in `_sequence_intrin_cache` (and in `sequence_intrinsics.csv`, which is re-used on later runs
when `REUSE_INTRINSICS_CSV = True`).

The depth formula for a vehicle bounding box `(x1, y1, x2, y2)`:

$$d_h = \frac{f_y \cdot H_{\mathrm{real}}}{y_2 - y_1}, \quad
  d_w = \frac{f_x \cdot W_{\mathrm{real}}}{x_2 - x_1}$$

**Focal-length policy.** DA3's predicted `fx` is ≈ 1.8 × `fy` on every drive, which is physically
impossible for square pixels (and makes width-based and height-based depths disagree by ~1.8×).
`INTRIN_FOCAL_POLICY` (config cell) selects how the pair is used; the raw predictions are always kept in
the `fx_raw / fy_raw` columns of `sequence_intrinsics.csv`.


In [ ]:
from collections import Counter
from dataclasses import replace as dc_replace


@dataclass
class CameraIntrinsics:
    fx:     float
    fy:     float
    cx:     float
    cy:     float
    fx_std: float = 0.0
    fy_std: float = 0.0
    source: str   = "assumed"
    # (fx, fy, fx_std, fy_std) exactly as predicted by DA3, i.e. before INTRIN_FOCAL_POLICY
    raw:    Optional[Tuple[float, float, float, float]] = None
    focal_policy: str = "as_predicted"

    @property
    def relative_uncertainty(self) -> float:
        return self.fx_std / (self.fx + 1e-6)

    @property
    def quality_weight(self) -> float:
        if self.source == "assumed_fov":
            return 0.60
        rel = self.relative_uncertainty
        return float(np.clip(1.0 - 10.0 * rel, 0.20, 1.0))

    def to_dict(self) -> dict:
        r = self.raw if self.raw is not None else (self.fx, self.fy, self.fx_std, self.fy_std)
        return dict(fx=self.fx, fy=self.fy, cx=self.cx, cy=self.cy,
                    fx_std=self.fx_std, fy_std=self.fy_std,
                    source=self.source, quality_weight=self.quality_weight,
                    fx_raw=r[0], fy_raw=r[1], fx_std_raw=r[2], fy_std_raw=r[3],
                    focal_policy=self.focal_policy)


def apply_focal_policy(cam: CameraIntrinsics, policy: Optional[str] = None) -> CameraIntrinsics:
    """Return a copy of `cam` whose fx/fy follow INTRIN_FOCAL_POLICY (always starts from raw values)."""
    policy = policy or INTRIN_FOCAL_POLICY
    fx, fy, fxs, fys = cam.raw if cam.raw is not None else (cam.fx, cam.fy, cam.fx_std, cam.fy_std)
    if policy == "as_predicted":
        nfx, nfy, nfxs, nfys = fx, fy, fxs, fys
    elif policy == "square_fy":
        nfx = nfy = fy;  nfxs = nfys = fys
    elif policy == "square_fx":
        nfx = nfy = fx;  nfxs = nfys = fxs
    elif policy == "square_mean":
        nfx = nfy = 0.5 * (fx + fy);  nfxs = nfys = 0.5 * (fxs + fys)
    else:
        raise ValueError(f"Unknown INTRIN_FOCAL_POLICY: {policy!r}")
    return dc_replace(cam, fx=float(nfx), fy=float(nfy), fx_std=float(nfxs), fy_std=float(nfys),
                      raw=(float(fx), float(fy), float(fxs), float(fys)), focal_policy=policy)


_intrin_model_cache = None
_sequence_intrin_cache: Dict[str, CameraIntrinsics] = {}


def _load_intrin_model():
    global _intrin_model_cache
    if _intrin_model_cache is not None:
        return _intrin_model_cache
    log.info("Loading DA3 intrinsic model: %s …", INTRIN_MODEL_ID)
    m = DepthAnything3.from_pretrained(INTRIN_MODEL_ID, cache_dir=CACHE_DIR)
    m = m.to(DEVICE).eval()
    _intrin_model_cache = m
    log.info("DA3 intrinsic model loaded.")
    return m


def _fallback_intrinsics(W: int, H: int, fov_deg: float = 65.0) -> CameraIntrinsics:
    fov_rad = np.radians(fov_deg)
    fx = (W / 2.0) / np.tan(fov_rad / 2.0)
    return CameraIntrinsics(fx=fx, fy=fx, cx=W / 2.0, cy=H / 2.0,
                             fx_std=fx * 0.10, fy_std=fx * 0.10,
                             source="assumed_fov")


def predict_sequence_intrinsics(image_paths: List[str],
                                 image_wh: Tuple[int, int]) -> CameraIntrinsics:
    W, H  = image_wh
    model = _load_intrin_model()
    chunks = [image_paths[i: i + INTRIN_CHUNK_SIZE]
              for i in range(0, len(image_paths), INTRIN_CHUNK_SIZE)]
    K_chunks: List[np.ndarray] = []
    for chunk in chunks:
        try:
            with torch.no_grad():
                pred = model.inference(
                    image=chunk,
                    process_res=INTRIN_PROCESS_RES,
                    process_res_method="upper_bound_resize",
                )
            K = pred.intrinsics
            if hasattr(K, "cpu"):
                K = K.cpu().numpy()
            K_chunks.append(np.asarray(K, dtype=np.float64))
        except Exception as exc:
            log.warning("Intrinsic chunk failed: %s", exc)

    if not K_chunks:
        log.warning("All intrinsic chunks failed — using FOV fallback.")
        return _fallback_intrinsics(W, H)

    all_K = np.concatenate(K_chunks, axis=0)
    K_avg = all_K.mean(axis=0)
    K_std = all_K.std(axis=0)

    # DA3 resizes isotropically (longest side -> INTRIN_PROCESS_RES), so one uniform factor.
    scale_factor = max(W, H) / INTRIN_PROCESS_RES
    K_avg[0, 0] *= scale_factor;  K_avg[1, 1] *= scale_factor
    K_avg[0, 2] *= scale_factor;  K_avg[1, 2] *= scale_factor
    K_std[0, 0] *= scale_factor;  K_std[1, 1] *= scale_factor

    cam_raw = CameraIntrinsics(
        fx=float(K_avg[0, 0]), fy=float(K_avg[1, 1]),
        cx=float(K_avg[0, 2]), cy=float(K_avg[1, 2]),
        fx_std=float(K_std[0, 0]), fy_std=float(K_std[1, 1]),
        source="da3_temporal",
        raw=(float(K_avg[0, 0]), float(K_avg[1, 1]), float(K_std[0, 0]), float(K_std[1, 1])),
    )
    cam = apply_focal_policy(cam_raw)
    log.info("Intrinsics: raw fx=%.1f fy=%.1f (fx/fy=%.2f)  → policy=%s → fx=%.1f fy=%.1f  quality=%.2f",
             cam_raw.fx, cam_raw.fy, cam_raw.fx / (cam_raw.fy + 1e-9),
             cam.focal_policy, cam.fx, cam.fy, cam.quality_weight)
    return cam


def get_sequence_intrinsics(seq_name: str,
                             seq_image_paths: List[str],
                             image_wh: Tuple[int, int]) -> CameraIntrinsics:
    if seq_name in _sequence_intrin_cache:
        return _sequence_intrin_cache[seq_name]
    log.info("Predicting intrinsics for: %s", seq_name)
    cam = predict_sequence_intrinsics(seq_image_paths, image_wh)
    _sequence_intrin_cache[seq_name] = cam
    return cam


def load_intrinsics_cache_from_csv(path: Optional[str] = None, overwrite: bool = False) -> int:
    """Fill _sequence_intrin_cache from sequence_intrinsics.csv (re-applying the CURRENT focal policy
    to the stored raw DA3 values).  Returns the number of drives loaded."""
    path = path or INTRINSICS_CSV
    if not os.path.exists(path):
        return 0
    df = pd.read_csv(path)

    def _num(rec, key, default):
        v = rec.get(key, default)
        return float(default) if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)

    n = 0
    for rec in df.to_dict("records"):
        drive = rec["drive"]
        if drive in _sequence_intrin_cache and not overwrite:
            continue
        raw = (_num(rec, "fx_raw", rec["fx"]),         _num(rec, "fy_raw", rec["fy"]),
               _num(rec, "fx_std_raw", rec["fx_std"]), _num(rec, "fy_std_raw", rec["fy_std"]))
        cam = CameraIntrinsics(fx=float(rec["fx"]), fy=float(rec["fy"]),
                               cx=float(rec["cx"]), cy=float(rec["cy"]),
                               fx_std=float(rec["fx_std"]), fy_std=float(rec["fy_std"]),
                               source=str(rec.get("source", "da3_temporal")), raw=raw)
        if cam.source == "da3_temporal":
            cam = apply_focal_policy(cam)
        _sequence_intrin_cache[drive] = cam
        n += 1
    return n

print("Camera intrinsic helpers defined.  Focal policy:", INTRIN_FOCAL_POLICY)


## Cell 5 · DA3METRIC-LARGE Depth Inference

Raw output is converted to orthographic metric depth (camera Z-axis, metres) via:

$$D_{\mathrm{metric}} = \frac{f_x^{\mathrm{KITTI}} \cdot D_{\mathrm{raw}}}{300}$$

Any systematic bias introduced by using the fixed KITTI focal length (rather than
the true per-sequence focal) is corrected by the vehicle-anchored scale/shift.

In [ ]:
_da3_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

_depth_model_cache = None


def load_depth_model():
    global _depth_model_cache
    if _depth_model_cache is not None:
        return _depth_model_cache
    log.info("Loading DA3METRIC-LARGE (%s) …", DEPTH_MODEL_ID)
    mdl = DepthAnything3.from_pretrained(
        DEPTH_MODEL_ID, cache_dir=CACHE_DIR
    ).to(DEVICE).eval()
    _depth_model_cache = mdl
    log.info("DA3 depth model ready on %s.", DEVICE)
    return mdl


def run_da3_depth(mdl, im_np: np.ndarray) -> np.ndarray:
    orig_h, orig_w = im_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)
    img_pil = Image.fromarray(im_np).resize((new_w, new_h), Image.BICUBIC)
    t_img   = _da3_transform(img_pil)
    inp     = t_img.unsqueeze(0).unsqueeze(1).to(DEVICE)
    with torch.no_grad():
        preds = mdl(inp, export_feat_layers=[])
        if isinstance(preds, dict):
            depth = preds["depth"]
        elif isinstance(preds, (tuple, list)):
            depth = preds[0]
        else:
            depth = preds
        if isinstance(depth, torch.Tensor):
            if depth.dim() == 3:
                depth = depth.unsqueeze(1)
            depth = (FOCAL_LENGTH_X * depth) / 300.0
            depth = F.interpolate(depth, size=(orig_h, orig_w),
                                   mode="bilinear", align_corners=False)
    return depth.squeeze().cpu().numpy().astype(np.float32)

print("DA3 depth model helpers defined.")

## Cell 6 · Pre-Generated Mask Loading  *(replaces SAM3 inference)*

Masks were generated by `inst_seg_notebook.ipynb` and stored under `MASK_ROOT`.  
The directory tree mirrors the KITTI layout exactly.

In [ ]:
def get_mask_dir(rgb_path: str) -> str:
    is_test      = "/test/" in rgb_path
    split_root   = TEST_ROOT      if is_test else TRAIN_ROOT
    mask_root    = MASK_ROOT_TEST if is_test else MASK_ROOT_TRAIN
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), split_root)
    return os.path.join(mask_root, rel_data_dir, frame_stem)


def load_masks_for_frame(mask_dir: str,
                           super_cat_filter: Optional[str] = None,
                           sub_cat_filter:   Optional[str] = None) -> List[dict]:
    mask_dir_path = Path(mask_dir)
    if not mask_dir_path.exists():
        return []
    frame_stem    = mask_dir_path.name
    manifest_path = mask_dir_path / f"{frame_stem}_manifest.json"

    if manifest_path.exists():
        with open(manifest_path, "r") as fh:
            records = json.load(fh)
        instances = []
        for rec in records:
            if super_cat_filter and rec["super_cat"] != super_cat_filter:
                continue
            if sub_cat_filter   and rec["sub_cat"]   != sub_cat_filter:
                continue
            mask_file = mask_dir_path / rec["mask_file"]
            if not mask_file.exists():
                continue
            mask_np = np.array(Image.open(mask_file).convert("L")) > 127
            instances.append({
                "uid":       rec["uid"],
                "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"],
                "sub_cat":   rec["sub_cat"],
                "score":     float(rec["score"]),
                "box":       rec.get("box"),
                "mask":      mask_np,
                "path":      mask_file,
            })
        return instances

    # Fallback: glob + filename parsing
    instances = []
    for mask_file in sorted(mask_dir_path.glob("*_mask.png")):
        stem   = mask_file.stem[:-5]
        if not stem.startswith(frame_stem + "_"):
            continue
        suffix = stem[len(frame_stem) + 1:]
        parts  = suffix.rsplit("_", 3)
        if len(parts) != 4:
            continue
        cat_str, idx_str, uid_str, score_str = parts
        cat_parts = cat_str.split("_", 1)
        if len(cat_parts) != 2:
            continue
        super_cat, sub_cat = cat_parts
        if super_cat_filter and super_cat != super_cat_filter:
            continue
        if sub_cat_filter   and sub_cat   != sub_cat_filter:
            continue
        try:
            local_idx = int(idx_str); score = float(score_str)
        except ValueError:
            continue
        mask_np = np.array(Image.open(mask_file).convert("L")) > 127
        instances.append({"uid": uid_str, "local_idx": local_idx,
                           "super_cat": super_cat, "sub_cat": sub_cat,
                           "score": score, "box": None,
                           "mask": mask_np, "path": mask_file})
    instances.sort(key=lambda x: x["local_idx"])
    return instances

print("Mask loading helpers defined.")

## Cell 7 · KITTI Dataset Indexing

In [ ]:
def index_kitti(use_train: bool = False) -> pd.DataFrame:
    rows = []
    def _collect(split_root: str, tag: str) -> None:
        pattern  = os.path.join(split_root, "*", "*_sync",
                                "proj_depth", "groundtruth", "image_02", "*.png")
        gt_paths = glob.glob(pattern)
        for gt_path in tqdm(gt_paths, desc=f"Indexing {tag}"):
            parts            = gt_path.split(os.sep)
            date, drive, fname = parts[-6], parts[-5], parts[-1]
            rgb_path = os.path.join(split_root, date, drive, "image_02", "data", fname)
            if not os.path.exists(rgb_path):
                continue
            rows.append(dict(folder=tag, sequence=date, drive=drive,
                             filename=fname, gt_path=gt_path, rgb_path=rgb_path))
    _collect(TEST_ROOT, "Test")
    if use_train:
        _collect(TRAIN_ROOT, "Train")
    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Dataset indexed: %d frames, %d drives.", len(df), df["drive"].nunique())
    return df


def get_sequence_image_paths(drive_root: str) -> List[str]:
    return sorted(glob.glob(os.path.join(drive_root, "image_02", "data", "*.png")))

print("KITTI indexing helpers defined.")

In [ ]:
# ── DEBUG: verify mask paths are resolving correctly ─────────────────────────
import os
from pathlib import Path

val_list_debug = index_kitti(use_train=USE_TRAIN)
sample = val_list_debug.iloc[0]
rgb_path = sample["rgb_path"]

frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
split_root   = TEST_ROOT if "/test/" in rgb_path else TRAIN_ROOT
rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), split_root)
mask_root    = MASK_ROOT_TEST if "/test/" in rgb_path else MASK_ROOT_TRAIN
mask_dir     = os.path.join(mask_root, rel_data_dir, frame_stem)

print(f"Sample rgb_path    : {rgb_path}")
print(f"split_root         : {split_root}")
print(f"rel_data_dir       : {rel_data_dir}")
print(f"Expected mask_dir  : {mask_dir}")
print(f"mask_dir exists    : {Path(mask_dir).exists()}")

# Show MASK_ROOT layout to compare
print(f"\nMASK_ROOT_TEST top-level contents:")
try:
    for entry in sorted(Path(MASK_ROOT_TEST).iterdir())[:5]:
        print(f"  {entry.name}/")
        for sub in sorted(entry.iterdir())[:3]:
            print(f"    {sub.name}/")
            for subsub in sorted(sub.iterdir())[:2]:
                print(f"      {subsub.name}/")
except Exception as e:
    print(f"  Could not list MASK_ROOT: {e}")


## Cell 8 · Vehicle Anchor Extraction

For each pre-generated vehicle mask:

1. Retrieve bounding box `(x1, y1, x2, y2)` from the manifest JSON (or compute
   from mask pixels as fallback).
2. Compute `pixel_height = y2 − y1`, `pixel_width = x2 − x1`.
3. Estimate metric depth:
   - From height: $d_h = f_y \cdot H_{\mathrm{real}} / (y_2 - y_1)$
   - From width:  $d_w = f_x \cdot W_{\mathrm{real}} / (x_2 - x_1)$
4. Consistency check: if $d_h / d_w > 1.3$ use the higher-weight estimate only.
5. Compute affine depth $a_i = \mathrm{median}(D_{\mathrm{raw}}[\mathrm{mask}])$.
6. Return anchor: $(a_i,\; d_i,\; w_i)$ where $w_i = r_{\mathrm{prior}} \cdot q_{\mathrm{intrin}} \cdot s_{\mathrm{SAM3}} \cdot \Sigma_j (1/\sigma_j^2)$.

In [ ]:
@dataclass
class VehicleAnchor:
    img_id:         int
    mask_uid:       str
    sub_cat:        str
    score:          float
    pixel_height:   float
    pixel_width:    float
    d_from_height:  Optional[float]
    d_from_width:   Optional[float]
    metric_depth:   float
    raw_da3_depth:  float
    anchor_weight:  float


def extract_vehicle_anchors(
    inst: dict,
    img_id: int,
    raw_depth: np.ndarray,
    intrinsics: CameraIntrinsics,
    image_hw: Tuple[int, int],
) -> Optional[VehicleAnchor]:
    sub_cat = inst["sub_cat"]
    prior   = VEHICLE_PRIORS.get(sub_cat)
    if prior is None:
        return None
    score = inst["score"]
    if score < MIN_MASK_SCORE:
        return None
    mask  = inst["mask"]
    H, W  = image_hw

    if mask.shape != (H, W):
        return None

    # ── Always derive tight bbox from mask pixels (ignores slack in manifest box) ──
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return None
    x1 = max(0, int(xs.min()));  x2 = min(W, int(xs.max()))
    y1 = max(0, int(ys.min()));  y2 = min(H, int(ys.max()))
    ph, pw = float(y2 - y1), float(x2 - x1)

    if ph < MIN_PIXEL_DIM and pw < MIN_PIXEL_DIM:
        return None

    # ── Depth sample: median of the raw DA3 depth inside the mask ────────────────
    mask_vals = raw_depth[mask]
    if mask_vals.size == 0:
        return None
    raw_med = float(np.median(mask_vals))

    # ── Orientation-aware dimension selection ────────────────────────────────────
    # Masks cover the FULL vehicle body.  When a car is side-on its pixel WIDTH
    # represents the car LENGTH (~4.5 m), not width (~1.8 m) — using it would
    # give a badly wrong depth.  Height is reliable at any orientation because
    # the roof-to-ground extent is stable regardless of yaw.
    #
    # aspect = pw/ph:  < 1.8 → frontal/rear-on (both dims usable)
    #                  1.8–2.5 → oblique (height only, down-weight width)
    #                  > 2.5 → side-on  (height only)
    aspect = pw / (ph + 1e-6)
    use_height = ph >= MIN_PIXEL_DIM
    if aspect < 1.8:
        use_width = pw >= MIN_PIXEL_DIM          # frontal/rear: width ≈ car width
        w_width_scale = 1.0
    elif aspect < 2.5:
        use_width = pw >= MIN_PIXEL_DIM          # oblique: allow but down-weight
        w_width_scale = 0.4
    else:
        use_width = False                        # side-on: width = car length, discard
        w_width_scale = 0.0

    depth_h_pair: Optional[Tuple[float, float]] = None
    depth_w_pair: Optional[Tuple[float, float]] = None

    if use_height and prior.real_height is not None:
        dh = intrinsics.fy * prior.real_height / ph
        if MIN_DEPTH < dh < MAX_DEPTH:
            depth_h_pair = (dh, prior.height_weight)

    if use_width and prior.real_width is not None:
        dw = intrinsics.fx * prior.real_width / pw
        if MIN_DEPTH < dw < MAX_DEPTH:
            depth_w_pair = (dw, prior.width_weight * w_width_scale)

    pairs = [p for p in [depth_h_pair, depth_w_pair] if p]
    if not pairs:
        return None

    depths  = np.array([p[0] for p in pairs])
    weights = np.array([p[1] for p in pairs])

    # Consistency check: if height/width estimates disagree >25%, keep the
    # higher-weight one only (tighter than old 1.30 threshold)
    if len(depths) == 2:
        ratio = max(depths) / (min(depths) + 1e-6)
        if ratio > 1.25:
            best    = int(np.argmax(weights))
            depths  = depths[best: best + 1]
            weights = weights[best: best + 1]

    metric_d  = float(np.average(depths, weights=weights))
    anchor_wt = prior.anchor_reliability * intrinsics.quality_weight * score * float(weights.sum())

    return VehicleAnchor(
        img_id=img_id, mask_uid=inst.get("uid", ""), sub_cat=sub_cat, score=score,
        pixel_height=ph, pixel_width=pw,
        d_from_height=depth_h_pair[0] if depth_h_pair else None,
        d_from_width=depth_w_pair[0]  if depth_w_pair  else None,
        metric_depth=metric_d, raw_da3_depth=raw_med, anchor_weight=anchor_wt,
    )

print("Vehicle anchor extraction defined.")

## Cell 8b · Vehicle-Mask Diagnostic  *(new)*

Answers three questions **before** the expensive passes:

1. Which vehicle `sub_cat` names exist in the mask store, and does each have a prior in `VEHICLE_PRIORS`?
2. What do the mask bounding boxes look like (pixel height/width, aspect ratio, fill ratio)?
3. Do the masks cover the **whole vehicle** (what the 1.50 m × 1.82 m priors assume) or only the lamps?
   → look at `plots/debug_vehicle_masks.png`.

Set `RUN_MASK_DIAGNOSTIC = False` in the config to skip it.


In [ ]:
from collections import Counter


def diagnose_vehicle_masks(val_list: pd.DataFrame,
                           n_frames: int = 60,
                           n_overlays: int = 12) -> Optional[pd.DataFrame]:
    idxs = np.linspace(0, len(val_list) - 1, min(n_frames, len(val_list))).astype(int)
    rows, overlays = [], []
    for idx in idxs:
        row   = val_list.iloc[int(idx)]
        insts = load_masks_for_frame(get_mask_dir(row["rgb_path"]), super_cat_filter="vehicle")
        for inst in insts:
            m = inst["mask"]
            ys, xs = np.where(m)
            if len(ys) == 0:
                continue
            ph = float(ys.max() - ys.min());  pw = float(xs.max() - xs.min())
            rows.append(dict(drive=row["drive"], frame=row["filename"], sub_cat=inst["sub_cat"],
                             score=inst["score"], has_prior=inst["sub_cat"] in VEHICLE_PRIORS,
                             px_h=ph, px_w=pw, aspect=pw / (ph + 1e-6),
                             fill=float(m.sum()) / ((ph + 1) * (pw + 1))))
            if len(overlays) < n_overlays and ph >= MIN_PIXEL_DIM and inst["score"] >= MIN_MASK_SCORE:
                overlays.append((row["rgb_path"], m, inst["sub_cat"], inst["score"]))

    if not rows:
        print("⚠  No vehicle masks found in the sampled frames — check MASK_ROOT_TEST / manifests.")
        return None

    d = pd.DataFrame(rows)
    g = d.groupby("sub_cat")
    summ = pd.DataFrame({
        "n_masks":    g.size(),
        "has_prior":  g["has_prior"].first(),
        "px_h_med":   g["px_h"].median(),
        "px_w_med":   g["px_w"].median(),
        "aspect_p10": g["aspect"].quantile(0.10),
        "aspect_med": g["aspect"].median(),
        "aspect_p90": g["aspect"].quantile(0.90),
        "fill_med":   g["fill"].median(),
    }).round(2)
    print("\nVehicle masks in the sampled frames (super_cat == 'vehicle'):")
    print(summ.to_string())

    missing = sorted(set(d.loc[~d["has_prior"], "sub_cat"]))
    if missing:
        print(f"\n⚠  sub_cats with NO entry in VEHICLE_PRIORS (they will never become anchors): {missing}")
    else:
        print("\n✓  Every vehicle sub_cat found has a prior.")

    print("\nHeuristic checks (whole-vehicle assumption, KITTI f≈720 px, car 1.5 m tall, 5–40 m away → ~27–216 px):")
    for sc, r in summ[summ["has_prior"]].iterrows():
        msgs = []
        if r["px_h_med"] < 15:
            msgs.append("median mask height < 15 px — masks look too SMALL for whole vehicles (lamps only?)")
        if r["aspect_med"] > 2.5:
            msgs.append("median aspect > 2.5 — mostly wide/side-on or lamp-pair masks; width prior will be discarded")
        if r["fill_med"] < 0.35:
            msgs.append("low bbox fill ratio — masks are sparse/fragmented")
        print(f"  {sc:<32} " + ("; ".join(msgs) if msgs else "OK — consistent with whole-vehicle masks"))

    if overlays:
        n = len(overlays);  ncol = 4;  nrow = int(np.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.2 * nrow))
        axes = np.atleast_1d(axes).ravel()
        for ax, (rgb_path, m, sc, score) in zip(axes, overlays):
            img = np.array(Image.open(rgb_path).convert("RGB"))
            ys, xs = np.where(m)
            y1, y2, x1, x2 = ys.min(), ys.max(), xs.min(), xs.max()
            my, mx = int(0.4 * (y2 - y1 + 1)) + 8, int(0.4 * (x2 - x1 + 1)) + 8
            cy1, cy2 = max(0, y1 - my), min(img.shape[0], y2 + my)
            cx1, cx2 = max(0, x1 - mx), min(img.shape[1], x2 + mx)
            crop = img[cy1:cy2, cx1:cx2].astype(np.float32)
            mm   = m[cy1:cy2, cx1:cx2]
            crop[mm] = 0.55 * crop[mm] + 0.45 * np.array([255, 0, 0], dtype=np.float32)
            ax.imshow(crop.astype(np.uint8))
            ax.add_patch(plt.Rectangle((x1 - cx1, y1 - cy1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor="yellow", linewidth=1.5))
            ax.set_title(f"{sc.replace('personal_vehicle_', '')}  s={score:.2f}  {int(x2-x1)}x{int(y2-y1)}px",
                         fontsize=8)
            ax.axis("off")
        for ax in axes[len(overlays):]:
            ax.axis("off")
        fig.suptitle("Vehicle masks (red) + tight bbox (yellow) — do they cover the WHOLE vehicle?", fontsize=11)
        plt.tight_layout()
        out = os.path.join(PLOT_DIR, "debug_vehicle_masks.png")
        fig.savefig(out, dpi=130, bbox_inches="tight");  plt.close(fig)
        print(f"\nOverlay montage saved → {out}")
    return summ


if RUN_MASK_DIAGNOSTIC:
    _vl = globals().get("val_list_debug")
    if _vl is None:
        _vl = index_kitti(use_train=USE_TRAIN)
    mask_diag_summary = diagnose_vehicle_masks(_vl, DIAG_N_FRAMES, DIAG_N_OVERLAYS)
else:
    print("RUN_MASK_DIAGNOSTIC=False — skipped.")


## Cell 9 · Global Correction Estimation  (RANSAC + WLS)

Fits the anchors $(a_i, d_i)$ — raw DA3 depth vs. depth implied by the vehicle size prior:

| `ONLY_SCALE` | Model | RANSAC sample | WLS refinement |
|---|---|---|---|
| `False` *(default)* | $d \approx s\,a + t$ | 2 anchors | weighted LS for $(s,t)$ on inliers |
| `True` | $d \approx s\,a$,  $t \equiv 0$ | 1 anchor | $s = \sum w\,a\,d \,/\, \sum w\,a^2$ on inliers |

* Inlier test: $|s\,a + t - d| <$ `RANSAC_INLIER_THR`; fitted $s$ must lie in `SCALE_BOUNDS`.
* If fewer than `MIN_ANCHORS_FOR_FIT` anchors exist the function **raises** (v5 silently returned scale 1 / shift 0).
* `report_anchor_sanity()` prints agreement diagnostics after the fit.
* The cache JSON stores `only_scale` and the focal policy; changing either and setting `SKIP_PASS1 = True`
  triggers an automatic cheap re-fit from `vehicle_anchors.csv`.


In [ ]:
@dataclass
class ScaleShiftResult:
    scale:      float
    shift:      float
    n_anchors:  int
    n_inliers:  int
    rmse_m:     float
    method:     str
    only_scale: bool = False


def estimate_global_scale_shift(anchors: List[VehicleAnchor],
                                only_scale: Optional[bool] = None) -> ScaleShiftResult:
    """Robust (RANSAC → WLS) fit of  metric ≈ scale·raw + shift.
    only_scale=True fixes shift = 0.00 and fits  metric ≈ scale·raw."""
    only_scale = ONLY_SCALE if only_scale is None else bool(only_scale)
    n = len(anchors)
    if n < MIN_ANCHORS_FOR_FIT:
        raise RuntimeError(
            f"Only {n} vehicle anchors (need ≥ {MIN_ANCHORS_FOR_FIT}). The vehicle-anchored fit cannot run.\n"
            f"  Most likely cause: VEHICLE_PRIORS keys do not match the mask sub_cat names "
            f"(run the mask-diagnostic cell), or the masks are being rejected "
            f"(MIN_MASK_SCORE / MIN_PIXEL_DIM / depth range).")

    raw = np.array([a.raw_da3_depth for a in anchors], dtype=np.float64)
    met = np.array([a.metric_depth  for a in anchors], dtype=np.float64)
    wts = np.array([a.anchor_weight for a in anchors], dtype=np.float64)
    wts = wts / (wts.sum() + 1e-12)
    lo, hi = SCALE_BOUNDS
    rng = np.random.default_rng(42)

    best_s, best_t, best_cnt = None, 0.0, -1
    for _ in range(RANSAC_ITERATIONS):
        if only_scale:
            i = int(rng.integers(n))
            if raw[i] < 1e-6:
                continue
            s, t = met[i] / raw[i], 0.0
        else:
            i, j = rng.choice(n, size=2, replace=False)
            dx = raw[j] - raw[i]
            if abs(dx) < 1e-3:                      # degenerate pair — slope undefined
                continue
            s = (met[j] - met[i]) / dx
            t = met[i] - s * raw[i]
        if not (lo <= s <= hi):
            continue
        cnt = int((np.abs(s * raw + t - met) < RANSAC_INLIER_THR).sum())
        if cnt > best_cnt:
            best_s, best_t, best_cnt = float(s), float(t), cnt

    method = "ransac_wls_scale_only" if only_scale else "ransac_wls_affine"
    if best_s is None:
        log.warning("RANSAC found no model inside SCALE_BOUNDS — using weighted-median ratio fallback.")
        order = np.argsort(met / np.maximum(raw, 1e-6))
        ratio = (met / np.maximum(raw, 1e-6))[order]
        cw    = np.cumsum(wts[order]);  best_s = float(ratio[np.searchsorted(cw, 0.5)])
        best_s = float(np.clip(best_s, lo, hi));  best_t = 0.0
        method = "median_fallback_scale_only" if only_scale else "median_fallback"

    # ── WLS refinement on the inliers (proper sqrt(w) weighting, no N×N matrix) ──
    inl = np.abs(best_s * raw + best_t - met) < RANSAC_INLIER_THR
    if inl.sum() >= (1 if only_scale else 2) and not method.startswith("median_fallback"):
        ri, mi, wi = raw[inl], met[inl], wts[inl]
        if only_scale:
            s_new = float(np.sum(wi * ri * mi) / (np.sum(wi * ri * ri) + 1e-12))
            t_new = 0.0
        else:
            sw = np.sqrt(wi)
            A  = np.vstack([ri, np.ones_like(ri)]).T * sw[:, None]
            try:
                p, *_ = np.linalg.lstsq(A, mi * sw, rcond=None)
                s_new, t_new = float(p[0]), float(p[1])
            except np.linalg.LinAlgError:
                s_new, t_new = best_s, best_t
        if lo <= s_new <= hi:
            best_s, best_t = s_new, t_new

    if only_scale:
        best_t = 0.0                                 # hard guarantee

    res  = np.abs(best_s * raw + best_t - met)
    inl  = res < RANSAC_INLIER_THR
    rmse = float(np.sqrt(np.mean(res[inl] ** 2))) if inl.any() else float("nan")

    log.info("Global scale=%.4f  shift=%.4f  inliers=%d/%d  RMSE=%.3fm  method=%s",
             best_s, best_t, int(inl.sum()), n, rmse, method)
    return ScaleShiftResult(scale=best_s, shift=best_t, n_anchors=n, n_inliers=int(inl.sum()),
                            rmse_m=rmse, method=method, only_scale=only_scale)


# ── Cache helpers ─────────────────────────────────────────────────────────────
def save_scale_cache(ss: ScaleShiftResult, path: Optional[str] = None) -> None:
    path = path or SCALE_CACHE
    payload = dict(scale=ss.scale, shift=ss.shift, n_anchors=ss.n_anchors, n_inliers=ss.n_inliers,
                   rmse_m=None if not np.isfinite(ss.rmse_m) else ss.rmse_m,
                   method=ss.method, only_scale=ss.only_scale,
                   intrin_focal_policy=INTRIN_FOCAL_POLICY)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    log.info("Scale/shift cached → %s", path)


def load_scale_cache(path: Optional[str] = None) -> dict:
    path = path or SCALE_CACHE
    if not os.path.exists(path):
        raise FileNotFoundError(f"SKIP_PASS1=True but no cache found at {path}")
    with open(path) as f:
        return json.load(f)


def cache_matches_config(cache: dict) -> bool:
    ok = (cache.get("method") not in (None, "no_fit")
          and cache.get("n_anchors", 0) >= MIN_ANCHORS_FOR_FIT
          and cache.get("only_scale") is not None
          and bool(cache.get("only_scale")) == bool(ONLY_SCALE)
          and cache.get("intrin_focal_policy") == INTRIN_FOCAL_POLICY)
    if ok and ONLY_SCALE and abs(cache.get("shift", 0.0)) > 0:
        ok = False
    return ok


def scale_result_from_cache(cache: dict) -> ScaleShiftResult:
    rm = cache.get("rmse_m")
    return ScaleShiftResult(scale=float(cache["scale"]), shift=float(cache["shift"]),
                            n_anchors=int(cache.get("n_anchors", 0)), n_inliers=int(cache.get("n_inliers", 0)),
                            rmse_m=float("nan") if rm is None else float(rm),
                            method=cache.get("method", "loaded"),
                            only_scale=bool(cache.get("only_scale", False)))


def load_anchors_from_csv(path: Optional[str] = None) -> List[VehicleAnchor]:
    """Rebuild the anchor list saved by Pass 1 (lets you re-fit with a different ONLY_SCALE for free)."""
    path = path or ANCHORS_CSV
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found — Pass 1 must be run once (SKIP_PASS1=False) to create the anchor list.")
    df = pd.read_csv(path)
    if "focal_policy" in df.columns:
        stale = sorted(set(df["focal_policy"].astype(str)) - {INTRIN_FOCAL_POLICY})
        if stale:
            raise RuntimeError(f"{path} was produced with focal policy {stale}, config says "
                               f"'{INTRIN_FOCAL_POLICY}'. Anchor depths depend on it → rerun Pass 1.")
    else:
        raise RuntimeError(f"{path} has no 'focal_policy' column (v5 file) → rerun Pass 1.")

    def _opt(v):
        return None if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)

    out = []
    for r in df.to_dict("records"):
        out.append(VehicleAnchor(
            img_id=int(r["img_id"]), mask_uid=str(r["mask_uid"]), sub_cat=str(r["sub_cat"]),
            score=float(r["score"]), pixel_height=float(r["pixel_height"]), pixel_width=float(r["pixel_width"]),
            d_from_height=_opt(r.get("d_from_height")), d_from_width=_opt(r.get("d_from_width")),
            metric_depth=float(r["metric_depth"]), raw_da3_depth=float(r["raw_da3_depth"]),
            anchor_weight=float(r["anchor_weight"])))
    log.info("Loaded %d anchors from %s", len(out), path)
    return out


# ── Post-fit sanity report ────────────────────────────────────────────────────
def report_anchor_sanity(anchors: List[VehicleAnchor], ss: ScaleShiftResult) -> None:
    raw = np.array([a.raw_da3_depth for a in anchors]);  met = np.array([a.metric_depth for a in anchors])
    fixed = ss.scale * raw + ss.shift
    ratio = met / np.maximum(raw, 1e-6)
    print("\n" + "─" * 78)
    print("ANCHOR SANITY REPORT")
    print("─" * 78)
    cnt = Counter(a.sub_cat for a in anchors)
    print("  anchors per sub_cat :", dict(cnt))
    print(f"  anchor metric depth : median {np.median(met):.2f} m  [{met.min():.1f} … {met.max():.1f}]")
    print(f"  raw DA3 depth       : median {np.median(raw):.2f} m  [{raw.min():.1f} … {raw.max():.1f}]")
    print(f"  metric / raw ratio  : median {np.median(ratio):.3f}  IQR [{np.percentile(ratio,25):.3f}, {np.percentile(ratio,75):.3f}]"
          "   (≈1.0–1.1 expected: DA3-metric is already near-metric on KITTI)")
    dh = np.array([a.d_from_height / max(a.raw_da3_depth, 1e-6) for a in anchors if a.d_from_height])
    dw = np.array([a.d_from_width  / max(a.raw_da3_depth, 1e-6) for a in anchors if a.d_from_width])
    if len(dh): print(f"  d_height / raw      : median {np.median(dh):.3f}   (n={len(dh)})")
    if len(dw): print(f"  d_width  / raw      : median {np.median(dw):.3f}   (n={len(dw)})")
    both = [a for a in anchors if a.d_from_height and a.d_from_width]
    if both:
        r_hw = np.array([a.d_from_height / a.d_from_width for a in both])
        print(f"  d_height / d_width  : median {np.median(r_hw):.3f}   (n={len(both)}; ≈1.0 when fx≈fy and prior sizes are right)")
    print(f"  after correction    : median bias {np.median(fixed - met):+.3f} m   MAE {np.mean(np.abs(fixed - met)):.3f} m")
    print(f"  model               : {'scale only (shift = 0.00)' if ss.only_scale else 'scale + shift'}  →  "
          f"scale={ss.scale:.4f}  shift={ss.shift:.4f}  inliers {ss.n_inliers}/{ss.n_anchors}")
    warns = []
    if not (0.5 <= np.median(ratio) <= 2.0):
        warns.append("anchor depths disagree strongly with DA3-metric depth (median ratio outside 0.5–2) → "
                     "check priors, intrinsics policy and whether masks cover the whole vehicle")
    lo, hi = SCALE_BOUNDS
    if ss.scale <= lo * 1.001 or ss.scale >= hi * 0.999:
        warns.append("fitted scale sits on the SCALE_BOUNDS limit")
    if ss.n_inliers < 0.4 * ss.n_anchors:
        warns.append("fewer than 40 % of anchors are inliers → priors/intrinsics are inconsistent")
    if len(dh) and len(dw) and abs(np.median(dh) / np.median(dw) - 1) > 0.25:
        warns.append("height-based and width-based depths differ by >25 % — focal policy or size priors suspect")
    for w in warns:
        print("  ⚠ ", w)
    print("─" * 78 + "\n")

print("Scale/shift estimation defined.  ONLY_SCALE =", ONLY_SCALE)


## Cell 10 · Evaluation Helpers  *(standard KITTI metrics)*

In [ ]:
def compute_depth_errors(gt: np.ndarray, pred: np.ndarray, mask: np.ndarray) -> dict:
    gt_v    = gt[mask]
    pred_v  = np.maximum(pred[mask], 1e-8)
    thresh  = np.maximum(gt_v / pred_v, pred_v / gt_v)
    return dict(
        rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2))),
        abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v)),
        sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / (gt_v ** 2))),
        rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2))),
        delta1   = float(np.mean(thresh < 1.25)),
        delta2   = float(np.mean(thresh < 1.25 ** 2)),
        delta3   = float(np.mean(thresh < 1.25 ** 3)),
    )


def eigen_crop_mask(shape: Tuple[int, ...]) -> np.ndarray:
    h, w = shape[:2]
    mask = np.zeros((h, w), dtype=bool)
    mask[44: min(44 + 352, h), 0:w] = True
    return mask


def compute_all_scales(gt_depth: np.ndarray, pred_raw: np.ndarray,
                        va_depth: np.ndarray, eval_mask: np.ndarray) -> dict:
    gt_v   = gt_depth[eval_mask]
    pred_c = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    results = {}

    # Unscaled
    m0 = compute_depth_errors(gt_depth, pred_c, eval_mask)
    results["unscaled"] = {"scale": 1.0, "shift": 0.0, "depth_map": pred_c, **m0,
                            "avg_depth": float(np.mean(pred_c[eval_mask])),
                            "min_depth": float(np.min(pred_c[eval_mask]))}

    # Median (oracle)
    s_med = float(np.median(gt_v) / np.median(pred_c[eval_mask]))
    p_med = np.clip(pred_c * s_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, p_med, eval_mask)
    results["median"] = {"scale": s_med, "shift": 0.0, "depth_map": p_med, **m1,
                          "avg_depth": float(np.mean(p_med[eval_mask])),
                          "min_depth": float(np.min(p_med[eval_mask]))}

    # Mean (oracle)
    s_mean = float(np.mean(gt_v) / np.mean(pred_c[eval_mask]))
    p_mean = np.clip(pred_c * s_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, p_mean, eval_mask)
    results["mean"] = {"scale": s_mean, "shift": 0.0, "depth_map": p_mean, **m2,
                        "avg_depth": float(np.mean(p_mean[eval_mask])),
                        "min_depth": float(np.min(p_mean[eval_mask]))}

    # Affine (oracle OLS)
    pf = pred_c[eval_mask].ravel()
    A_mat = np.vstack([pf, np.ones_like(pf)]).T
    par, *_ = np.linalg.lstsq(A_mat, gt_v.ravel(), rcond=None)
    s_aff, t_aff = float(par[0]), float(par[1])
    p_aff = np.clip(s_aff * pred_c + t_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, p_aff, eval_mask)
    results["affine"] = {"scale": s_aff, "shift": t_aff, "depth_map": p_aff, **m3,
                          "avg_depth": float(np.mean(p_aff[eval_mask])),
                          "min_depth": float(np.min(p_aff[eval_mask]))}

    # Vehicle-anchored (ours)
    va_c = np.clip(va_depth, MIN_DEPTH, MAX_DEPTH)
    m4 = compute_depth_errors(gt_depth, va_c, eval_mask)
    results["va"] = {"scale": None, "shift": None, "depth_map": va_c, **m4,
                      "avg_depth": float(np.mean(va_c[eval_mask])),
                      "min_depth": float(np.min(va_c[eval_mask]))}

    return results

print("Evaluation helpers defined.")

## Cell 11 · CSV Column Schema

In [ ]:
CSV_COLUMNS = [
    "img_id","file_name","mask_id","mask_uid","mask_score","super_cat","sub_cat","horizontal_angle",
    "pixel_height","pixel_width","d_from_height","d_from_width",
    "anchor_metric_depth","anchor_raw_depth","anchor_weight",
    "unscaled_rmse","unscaled_abs_rel","unscaled_sq_rel","unscaled_rmse_log",
    "unscaled_delta1","unscaled_delta2","unscaled_delta3","unscaled_avg_depth","unscaled_min_depth",
    "median_scale","median_rmse","median_abs_rel","median_sq_rel","median_rmse_log",
    "median_delta1","median_delta2","median_delta3","median_avg_depth","median_min_depth",
    "mean_scale","mean_rmse","mean_abs_rel","mean_sq_rel","mean_rmse_log",
    "mean_delta1","mean_delta2","mean_delta3","mean_avg_depth","mean_min_depth",
    "affine_scale","affine_shift","affine_rmse","affine_abs_rel","affine_sq_rel","affine_rmse_log",
    "affine_delta1","affine_delta2","affine_delta3","affine_avg_depth","affine_min_depth",
    "va_global_scale","va_global_shift","va_rmse","va_abs_rel","va_sq_rel","va_rmse_log",
    "va_delta1","va_delta2","va_delta3","va_avg_depth","va_min_depth",
    "lidar_avg_depth","lidar_min_depth",
    "rgb_image_path","gt_depth_map_path","da3_pred_map_path","mask_path",
]
print(f"CSV_COLUMNS defined ({len(CSV_COLUMNS)} columns).")

## Cell 12 · Pass 1 — Vehicle Anchor Collection

Iterates over all frames, runs DA3 depth inference and collects vehicle anchors.
Intrinsics are predicted **once per sequence/drive** using DA3 and cached.

In [ ]:
def run_pass1_collect_anchors(val_list: pd.DataFrame, da3_mdl) -> List[VehicleAnchor]:
    all_anchors: List[VehicleAnchor] = []
    anchor_rows: List[dict] = []

    if REUSE_INTRINSICS_CSV:
        n_loaded = load_intrinsics_cache_from_csv()
        if n_loaded:
            log.info("Re-using intrinsics for %d drives from %s (policy=%s).",
                     n_loaded, INTRINSICS_CSV, INTRIN_FOCAL_POLICY)

    # Pre-index image paths per drive for intrinsic prediction
    drive_to_paths: Dict[str, List[str]] = {}
    for row in val_list.itertuples():
        drive_root = os.path.join(TEST_ROOT, row.sequence, row.drive)
        if row.drive not in drive_to_paths:
            drive_to_paths[row.drive] = get_sequence_image_paths(drive_root)

    frames_no_masks = frames_no_usable = n_rejected = 0
    seen_sub_cats: Counter = Counter()
    unmatched_sub_cats: Counter = Counter()

    for index, row in tqdm(val_list.iterrows(), total=len(val_list),
                            desc="Pass 1 — Anchor Collection"):
        if index < START_INDEX:
            continue
        try:
            mask_dir = get_mask_dir(row["rgb_path"])
            vehicle_insts = load_masks_for_frame(mask_dir, super_cat_filter="vehicle")
            if not vehicle_insts:
                frames_no_masks += 1
                continue
            for inst in vehicle_insts:
                seen_sub_cats[inst["sub_cat"]] += 1
                if inst["sub_cat"] not in VEHICLE_PRIORS:
                    unmatched_sub_cats[inst["sub_cat"]] += 1
            usable = [i for i in vehicle_insts
                      if i["sub_cat"] in VEHICLE_PRIORS and i["score"] >= MIN_MASK_SCORE]
            if not usable:
                frames_no_usable += 1          # no DA3 inference wasted on this frame
                continue

            image_np  = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            H, W, _   = image_np.shape
            seq_paths = drive_to_paths.get(row["drive"], [])
            cam       = get_sequence_intrinsics(row["drive"], seq_paths, (W, H))
            raw_depth = run_da3_depth(da3_mdl, image_np)
            for inst in usable:
                anchor = extract_vehicle_anchors(inst, int(index), raw_depth, cam, (H, W))
                if anchor is None:
                    n_rejected += 1
                    continue
                all_anchors.append(anchor)
                anchor_rows.append({
                    "img_id": index, "drive": row["drive"], "filename": row["filename"],
                    "mask_uid": anchor.mask_uid, "sub_cat": anchor.sub_cat,
                    "score": round(anchor.score, 4),
                    "pixel_height": round(anchor.pixel_height, 2),
                    "pixel_width":  round(anchor.pixel_width,  2),
                    "d_from_height": round(anchor.d_from_height, 4) if anchor.d_from_height else None,
                    "d_from_width":  round(anchor.d_from_width,  4) if anchor.d_from_width  else None,
                    "metric_depth":  round(anchor.metric_depth,  4),
                    "raw_da3_depth": round(anchor.raw_da3_depth, 4),
                    "anchor_weight": round(anchor.anchor_weight, 6),
                    "fx": round(cam.fx, 2), "fy": round(cam.fy, 2),
                    "intrin_source": cam.source,
                    "intrin_quality": round(cam.quality_weight, 4),
                    "focal_policy": cam.focal_policy,
                })
        except Exception as exc:
            log.error("Pass 1 error at index %d: %s", index, exc, exc_info=True)

    log.info("Pass 1 complete: %d anchors | frames: %d without vehicle masks, %d with none usable | "
             "%d usable masks rejected by extract_vehicle_anchors.",
             len(all_anchors), frames_no_masks, frames_no_usable, n_rejected)
    log.info("Vehicle sub_cats seen in masks: %s", dict(seen_sub_cats))
    if unmatched_sub_cats:
        log.warning("These vehicle sub_cats have NO prior in VEHICLE_PRIORS and were ignored: %s",
                    dict(unmatched_sub_cats))

    if anchor_rows:
        pd.DataFrame(anchor_rows).to_csv(ANCHORS_CSV, index=False)
        log.info("Anchors CSV saved → %s", ANCHORS_CSV)

    if _sequence_intrin_cache:
        pd.DataFrame([{"drive": d, **c.to_dict()}
                       for d, c in _sequence_intrin_cache.items()]
        ).to_csv(INTRINSICS_CSV, index=False)
        log.info("Intrinsics CSV saved → %s", INTRINSICS_CSV)

    return all_anchors

print("run_pass1_collect_anchors() defined.")


## Cell 13 · Pass 2 — Evaluation

Applies the global vehicle-anchored (scale, shift) to raw DA3 depth, then
computes all five scale variants vs LiDAR GT for every mask in every frame.

In [ ]:
def run_pass2_evaluate(val_list: pd.DataFrame, da3_mdl,
                        va_scale: float, va_shift: float) -> None:
    RGB_DIR  = os.path.join(MAIN_OUTPUT_DIR, "rgb_images");  os.makedirs(RGB_DIR, exist_ok=True)
    GT_DIR   = os.path.join(MAIN_OUTPUT_DIR, "gt_depth");    os.makedirs(GT_DIR,  exist_ok=True)
    DA3_DIR  = os.path.join(MAIN_OUTPUT_DIR, "da3_pred");    os.makedirs(DA3_DIR, exist_ok=True)
    MASK_OUT = os.path.join(MAIN_OUTPUT_DIR, "masks");       os.makedirs(MASK_OUT,exist_ok=True)

    # ── sanity guards ──────────────────────────────────────────────────────────
    if ONLY_SCALE and abs(va_shift) > 0.0:
        raise ValueError(f"ONLY_SCALE=True but va_shift={va_shift} ≠ 0 — cache/config mismatch.")
    if abs(va_scale - 1.0) < 1e-9 and abs(va_shift) < 1e-9:
        log.warning("VA correction is the identity (scale=1, shift=0): the 'va_*' columns will be "
                    "identical to 'unscaled_*'. This should only happen if the fit really returned 1/0.")

    # v5 appended to an existing file, duplicating every row on re-runs → start fresh by default
    if (OVERWRITE_RESULTS and START_INDEX == 0) or not os.path.exists(OUTPUT_CSV_PATH):
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(OUTPUT_CSV_PATH, index=False)

    # intrinsics are needed for the per-mask anchor diagnostics (empty when SKIP_PASS1=True)
    if not _sequence_intrin_cache:
        n_i = load_intrinsics_cache_from_csv()
        log.info("Loaded intrinsics for %d drives from %s for anchor diagnostics.", n_i, INTRINSICS_CSV)

    frames_no_masks = 0

    for index, row in tqdm(val_list.iterrows(), total=len(val_list), desc="Pass 2 — Evaluation"):
        if index < START_INDEX:
            continue
        base_fn = row["folder"] + "_" + row["drive"] + "_" + os.path.splitext(row["filename"])[0]
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = np.array(Image.open(row["gt_path"])).astype(np.uint16) / 256.0
            H, W, _  = image_np.shape
            cx_img   = W / 2.0

            mask_dir      = get_mask_dir(row["rgb_path"])
            all_instances = load_masks_for_frame(mask_dir)
            if not all_instances:
                frames_no_masks += 1
                continue

            pred_raw = run_da3_depth(da3_mdl, image_np)
            va_depth = np.clip(va_scale * pred_raw + va_shift, MIN_DEPTH, MAX_DEPTH)

            rgb_out = os.path.join(RGB_DIR, f"{base_fn}.png")
            gt_out  = os.path.join(GT_DIR,  f"{base_fn}.npy")
            da3_out = os.path.join(DA3_DIR, f"{base_fn}.npy")
            Image.fromarray(image_np).save(rgb_out)
            if SAVE_NPY:
                np.save(gt_out, gt_depth); np.save(da3_out, pred_raw)

            crop_mask = eigen_crop_mask(gt_depth.shape)

            for inst in all_instances:
                seg_mask  = inst["mask"]; score = inst["score"]
                super_cat = inst["super_cat"]; sub_cat = inst["sub_cat"]
                mask_idx  = inst["local_idx"]; mask_uid = inst.get("uid", "")

                if score < MIN_MASK_SCORE:
                    continue
                if seg_mask.shape != (H, W):
                    continue

                eval_mask = seg_mask & (gt_depth > MIN_DEPTH) & (gt_depth < MAX_DEPTH) & crop_mask
                if not np.any(eval_mask):
                    continue

                # Anchor diagnostics (vehicle only) — SAME code path as Pass 1
                anch_h = anch_w = anch_dh = anch_dw = anch_met = anch_raw = anch_wt = None
                if super_cat == "vehicle" and sub_cat in VEHICLE_PRIORS:
                    cam = _sequence_intrin_cache.get(row["drive"])
                    if cam is not None:
                        anc = extract_vehicle_anchors(inst, int(index), pred_raw, cam, (H, W))
                        if anc is not None:
                            anch_h   = round(anc.pixel_height, 2)
                            anch_w   = round(anc.pixel_width, 2)
                            anch_dh  = round(anc.d_from_height, 4) if anc.d_from_height else None
                            anch_dw  = round(anc.d_from_width, 4)  if anc.d_from_width  else None
                            anch_met = round(anc.metric_depth, 4)
                            anch_raw = round(anc.raw_da3_depth, 4)
                            anch_wt  = round(anc.anchor_weight, 6)

                scale_res  = compute_all_scales(gt_depth, pred_raw, va_depth, eval_mask)
                lidar_vals = gt_depth[eval_mask]
                ys_seg, xs_seg = np.where(seg_mask)
                cx2 = float(np.mean(xs_seg)) if len(xs_seg) else cx_img
                angle_deg = math.degrees(math.atan((cx2 - cx_img) / FOCAL_LENGTH_X))

                mask_path_out = str(inst.get("path",""))
                if SAVE_NPY and mask_path_out:
                    np.save(os.path.join(MASK_OUT, f"{base_fn}_mask_{mask_idx}.npy"), eval_mask)

                def _m(v,k): return scale_res[v].get(k)
                csv_row = {
                    "img_id":index,"file_name":base_fn,"mask_id":mask_idx,"mask_uid":mask_uid,
                    "mask_score":round(score,4),"super_cat":super_cat,"sub_cat":sub_cat,
                    "horizontal_angle":round(angle_deg,4),
                    "pixel_height":anch_h,"pixel_width":anch_w,"d_from_height":anch_dh,
                    "d_from_width":anch_dw,"anchor_metric_depth":anch_met,"anchor_raw_depth":anch_raw,
                    "anchor_weight":anch_wt,
                    "unscaled_rmse":_m("unscaled","rmse"),"unscaled_abs_rel":_m("unscaled","abs_rel"),
                    "unscaled_sq_rel":_m("unscaled","sq_rel"),"unscaled_rmse_log":_m("unscaled","rmse_log"),
                    "unscaled_delta1":_m("unscaled","delta1"),"unscaled_delta2":_m("unscaled","delta2"),
                    "unscaled_delta3":_m("unscaled","delta3"),"unscaled_avg_depth":_m("unscaled","avg_depth"),
                    "unscaled_min_depth":_m("unscaled","min_depth"),
                    "median_scale":_m("median","scale"),"median_rmse":_m("median","rmse"),
                    "median_abs_rel":_m("median","abs_rel"),"median_sq_rel":_m("median","sq_rel"),
                    "median_rmse_log":_m("median","rmse_log"),"median_delta1":_m("median","delta1"),
                    "median_delta2":_m("median","delta2"),"median_delta3":_m("median","delta3"),
                    "median_avg_depth":_m("median","avg_depth"),"median_min_depth":_m("median","min_depth"),
                    "mean_scale":_m("mean","scale"),"mean_rmse":_m("mean","rmse"),
                    "mean_abs_rel":_m("mean","abs_rel"),"mean_sq_rel":_m("mean","sq_rel"),
                    "mean_rmse_log":_m("mean","rmse_log"),"mean_delta1":_m("mean","delta1"),
                    "mean_delta2":_m("mean","delta2"),"mean_delta3":_m("mean","delta3"),
                    "mean_avg_depth":_m("mean","avg_depth"),"mean_min_depth":_m("mean","min_depth"),
                    "affine_scale":_m("affine","scale"),"affine_shift":_m("affine","shift"),
                    "affine_rmse":_m("affine","rmse"),"affine_abs_rel":_m("affine","abs_rel"),
                    "affine_sq_rel":_m("affine","sq_rel"),"affine_rmse_log":_m("affine","rmse_log"),
                    "affine_delta1":_m("affine","delta1"),"affine_delta2":_m("affine","delta2"),
                    "affine_delta3":_m("affine","delta3"),"affine_avg_depth":_m("affine","avg_depth"),
                    "affine_min_depth":_m("affine","min_depth"),
                    "va_global_scale":round(va_scale,6),"va_global_shift":round(va_shift,6),
                    "va_rmse":_m("va","rmse"),"va_abs_rel":_m("va","abs_rel"),
                    "va_sq_rel":_m("va","sq_rel"),"va_rmse_log":_m("va","rmse_log"),
                    "va_delta1":_m("va","delta1"),"va_delta2":_m("va","delta2"),
                    "va_delta3":_m("va","delta3"),"va_avg_depth":_m("va","avg_depth"),
                    "va_min_depth":_m("va","min_depth"),
                    "lidar_avg_depth":round(float(np.mean(lidar_vals)),4),
                    "lidar_min_depth":round(float(np.min(lidar_vals)),4),
                    "rgb_image_path":rgb_out,"gt_depth_map_path":gt_out,
                    "da3_pred_map_path":da3_out,"mask_path":mask_path_out,
                }
                pd.DataFrame([csv_row]).to_csv(OUTPUT_CSV_PATH, mode="a", header=False, index=False)

        except Exception as exc:
            log.error("Pass 2 error at index %d (%s): %s", index, base_fn, exc, exc_info=True)

    log.info("Pass 2 complete → %s", OUTPUT_CSV_PATH)

print("run_pass2_evaluate() defined.")

## Cell 14 · Analysis & Visualisation Helpers

In [ ]:
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
    "va":       "#e67e22",
}
SCALE_LABELS = {
    "unscaled": "Unscaled",
    "median":   "Median Scale (oracle)",
    "mean":     "Mean Scale (oracle)",
    "affine":   "Affine Scale (oracle)",
    "va":       "Vehicle-Anchored (ours, scale-only)" if ONLY_SCALE else "Vehicle-Anchored (ours)",
}
ALL_KEYS = list(SCALE_COLORS.keys())


def _save_fig(fig: plt.Figure, name: str, dpi: int = 150) -> None:
    os.makedirs(PLOT_DIR, exist_ok=True)
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    log.info("  ✓  %s", name)


def load_results() -> pd.DataFrame:
    df = pd.read_csv(OUTPUT_CSV_PATH)
    df = df.dropna(subset=["unscaled_rmse","median_rmse","mean_rmse",
                             "affine_rmse","va_rmse","lidar_avg_depth"])
    return df

print("Analysis colour / label config and helpers defined.")

## Cell 15 · Plot Functions  *(13 plots)*

In [ ]:
def plot_gt_vs_pred(df: pd.DataFrame) -> None:
    variants = [
        ("Unscaled",              "lidar_avg_depth", "unscaled_avg_depth"),
        ("Median Scale (oracle)", "lidar_avg_depth", "median_avg_depth"),
        ("Affine Scale (oracle)", "lidar_avg_depth", "affine_avg_depth"),
        ("Vehicle-Anchored",      "lidar_avg_depth", "va_avg_depth"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    colors = [SCALE_COLORS["unscaled"], SCALE_COLORS["median"],
              SCALE_COLORS["affine"],   SCALE_COLORS["va"]]
    for ax, (title, xcol, ycol), color in zip(axes, variants, colors):
        sns.regplot(ax=ax, data=df, x=xcol, y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_title(f"DA3METRIC-LARGE  |  {title}", fontsize=11)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH); ax.legend(fontsize=9)
    fig.suptitle("Ground-Truth vs Predicted Depth — Key Scale Variants", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, "01_gt_vs_pred_all_scales.png")


# ── Plot 02: GT Depth vs RMSE ─────────────────────────────────────────────────
def plot_gt_vs_rmse(df: pd.DataFrame) -> None:
    variants = [(SCALE_LABELS[k], f"{k}_rmse", SCALE_COLORS[k]) for k in ALL_KEYS]
    ymax = df[[f"{k}_rmse" for k in ALL_KEYS]].max().max() * 1.12
    fig, axes = plt.subplots(2, 3, figsize=(21, 12)); axes = axes.flatten()
    for ax, (title, ycol, color) in zip(axes, variants):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"{title}", fontsize=11)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("RMSE (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, ymax)
    for ax in axes[len(variants):]:
        ax.set_visible(False)
    fig.suptitle("GT Depth vs RMSE — All Scale Variants", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, "02_gt_vs_rmse_all_scales.png")


# ── Plot 03: Horizontal Angle vs RMSE ────────────────────────────────────────
def plot_angle_vs_rmse(df: pd.DataFrame) -> None:
    variants = [(SCALE_LABELS[k], f"{k}_rmse", SCALE_COLORS[k]) for k in ALL_KEYS]
    ymax_raw = df[[f"{k}_rmse" for k in ALL_KEYS]].max().max()
    ymax = ymax_raw * 1.12 if np.isfinite(ymax_raw) and ymax_raw > 0 else 1.0
    fig, axes = plt.subplots(2, 3, figsize=(21, 12)); axes = axes.flatten()
    for ax, (title, ycol, color) in zip(axes, variants):
        sns.regplot(ax=ax, data=df, x="horizontal_angle", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"{title}: Angle vs RMSE", fontsize=11)
        ax.set_xlabel("Horizontal Angle (°)"); ax.set_ylabel("RMSE (m)"); ax.set_ylim(0, ymax)
    for ax in axes[len(variants):]:
        ax.set_visible(False)
    fig.suptitle("Horizontal Angle vs RMSE — All Scale Variants", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, "03_angle_vs_rmse_all_scales.png")


# ── Plot 04: Scale Method Comparison Bar Chart ────────────────────────────────
def plot_scale_comparison_bars(df: pd.DataFrame) -> None:
    metrics_spec = [
        ("RMSE (m)  ↓", [f"{k}_rmse"    for k in ALL_KEYS]),
        ("Abs Rel   ↓", [f"{k}_abs_rel"  for k in ALL_KEYS]),
        ("δ < 1.25  ↑", [f"{k}_delta1"  for k in ALL_KEYS]),
    ]
    labels = [SCALE_LABELS[k] for k in ALL_KEYS]
    colors = [SCALE_COLORS[k] for k in ALL_KEYS]
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    for ax, (metric_name, cols) in zip(axes, metrics_spec):
        vals = [df[c].mean() for c in cols]
        bars = ax.bar(labels, vals, color=colors, edgecolor="white", width=0.6)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
        ax.set_title(metric_name, fontsize=12); ax.set_ylabel(metric_name)
        ax.set_ylim(0, max(vals) * 1.25); ax.grid(axis="y", alpha=0.35)
        ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9)
    fig.suptitle("DA3METRIC-LARGE — Scale Method Comparison", fontsize=14, y=1.02)
    plt.tight_layout(); _save_fig(fig, "04_scale_comparison_bars.png")


# ── Plot 05: Predicted vs LiDAR by Object Category ───────────────────────────
def plot_model_vs_lidar_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    n = max(len(cats), 1)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 6))
    if n == 1:
        axes = [axes]
    for ax, cat in zip(axes, cats):
        sub = df[df["super_cat"] == cat]
        for key in ALL_KEYS:
            ax.scatter(sub["lidar_avg_depth"], sub[f"{key}_avg_depth"],
                       s=18, alpha=0.4, c=SCALE_COLORS[key], label=SCALE_LABELS[key])
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_title(f"Category: {cat}", fontsize=12); ax.legend(fontsize=7)
    fig.suptitle("Predicted Depth vs LiDAR GT — by Object Category", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "05_model_vs_lidar_by_category.png")


# ── Plot 06: RMSE Distribution Histograms ─────────────────────────────────────
def plot_rmse_distributions(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(21, 12)); axes = axes.flatten()
    for ax, key in zip(axes, ALL_KEYS):
        col  = f"{key}_rmse"
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=SCALE_COLORS[key], alpha=0.72, edgecolor="white")
        ax.axvline(vals.mean(),   color="black", linestyle="--", linewidth=1.8,
                   label=f"Mean   = {vals.mean():.3f} m")
        ax.axvline(vals.median(), color="grey",  linestyle=":",  linewidth=1.8,
                   label=f"Median = {vals.median():.3f} m")
        ax.set_title(f"RMSE Distribution — {SCALE_LABELS[key]}", fontsize=11)
        ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    for ax in axes[len(ALL_KEYS):]:
        ax.set_visible(False)
    fig.suptitle("RMSE Distributions Across All Masks", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "06_rmse_distributions.png")


# ── Plot 07: Scale Factor Distributions ──────────────────────────────────────
def plot_scale_distributions(df: pd.DataFrame) -> None:
    spec = [
        ("median_scale", "Median Scale (oracle)", SCALE_COLORS["median"]),
        ("mean_scale",   "Mean Scale (oracle)",   SCALE_COLORS["mean"]),
        ("affine_scale", "Affine Scale (oracle)", SCALE_COLORS["affine"]),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        va_s = df["va_global_scale"].iloc[0] if "va_global_scale" in df else None
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1.8, label="Scale = 1.0")
        ax.axvline(vals.mean(), color="red", linestyle=":", linewidth=1.8,
                   label=f"Dataset Mean = {vals.mean():.3f}")
        if va_s is not None:
            ax.axvline(va_s, color=SCALE_COLORS["va"], linestyle="-.",
                       linewidth=2.0, label=f"VA Scale = {va_s:.3f}")
        ax.set_title(f"{label} Distribution", fontsize=11)
        ax.set_xlabel("Scale Factor"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor Distributions", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "07_scale_distributions.png")


# ── Plot 08: 3D Error Surfaces ────────────────────────────────────────────────
def plot_3d_error_surfaces(df: pd.DataFrame, degree: int = 2) -> None:
    keys    = ["unscaled", "affine", "va"]
    colors  = [SCALE_COLORS[k] for k in keys]
    labels  = [SCALE_LABELS[k] for k in keys]
    xcol, ycol = "lidar_avg_depth", "horizontal_angle"
    df_c = df.dropna(subset=[xcol, ycol] + [f"{k}_rmse" for k in keys])
    if df_c.empty:
        return
    x_min, x_max = df_c[xcol].min(), df_c[xcol].max()
    y_min, y_max = df_c[ycol].min(), df_c[ycol].max()
    z_vals = [df_c[f"{k}_rmse"].values for k in keys]
    z_min  = min(z.min() for z in z_vals)
    z_max  = max(z.max() for z in z_vals)
    buf = lambda r: r * 0.08
    fig = plt.figure(figsize=(22, 6))
    fig.suptitle(f"3D Error Analysis (poly deg={degree}) — key variants", fontsize=13)
    for sp_idx, (key, color, label) in enumerate(zip(keys, colors, labels), start=1):
        ax = fig.add_subplot(1, 3, sp_idx, projection="3d")
        zcol = f"{key}_rmse"
        ax.scatter(df_c[xcol], df_c[ycol], df_c[zcol], c=color, alpha=0.35, s=8)
        Xf = df_c[[xcol, ycol]].values; Zf = df_c[zcol].values
        pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        pipe.fit(Xf, Zf)
        xs, ys = np.meshgrid(np.linspace(x_min, x_max, 40),
                              np.linspace(y_min, y_max, 40))
        zs = pipe.predict(np.c_[xs.ravel(), ys.ravel()]).reshape(xs.shape)
        ax.plot_surface(xs, ys, zs, color=color, alpha=0.22, rstride=1, cstride=1)
        ax.set_xlabel("GT Depth (m)", fontsize=8); ax.set_ylabel("H. Angle (°)", fontsize=8)
        ax.set_zlabel("RMSE (m)", fontsize=8); ax.set_title(label, fontsize=10)
        bx, by, bz = buf(x_max-x_min), buf(y_max-y_min), buf(z_max-z_min)
        ax.set_xlim(x_min-bx, x_max+bx); ax.set_ylim(y_min-by, y_max+by)
        ax.set_zlim(z_min-bz, z_max+bz)
    plt.tight_layout(); _save_fig(fig, "08_3d_error_surfaces.png")


# ── Plot 09: Depth-Bin Bias Analysis ─────────────────────────────────────────
def plot_depth_bin_analysis(df: pd.DataFrame) -> None:
    bins   = np.arange(0, MAX_DEPTH + 10, 10)
    labels = [f"{int(b)}–{int(b+10)}m" for b in bins[:-1]]
    df2    = df.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_avg_depth"], bins=bins, labels=labels)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    ax = axes[0]
    for key in ALL_KEYS:
        agg = df2.groupby("depth_bin", observed=True)[f"{key}_rmse"].mean()
        ax.plot(agg.index, agg.values, "o-",
                color=SCALE_COLORS[key], label=SCALE_LABELS[key], linewidth=2)
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE by Depth Bin")
    ax.legend(fontsize=8); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    ax = axes[1]
    for key in ALL_KEYS:
        agg_pred  = df2.groupby("depth_bin", observed=True)[f"{key}_avg_depth"].mean()
        agg_lidar = df2.groupby("depth_bin", observed=True)["lidar_avg_depth"].mean()
        bias = agg_pred - agg_lidar
        ax.plot(bias.index, bias.values, "o-",
                color=SCALE_COLORS[key], label=SCALE_LABELS[key], linewidth=2)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.5, label="Zero bias")
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Prediction Bias (m)")
    ax.set_title("Depth Bias by Bin (Pred − LiDAR)")
    ax.legend(fontsize=8); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    fig.suptitle("Depth-Bin Analysis — DA3METRIC-LARGE", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "09_depth_bin_analysis.png")


# ── Plot 10: RMSE per Object Category ────────────────────────────────────────
def plot_rmse_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    if len(cats) < 1:
        return
    n_keys = len(ALL_KEYS)
    x = np.arange(len(cats))
    w = 0.15
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, key in enumerate(ALL_KEYS):
        vals   = [df[df["super_cat"] == c][f"{key}_rmse"].mean() for c in cats]
        offset = (i - (n_keys - 1) / 2.0) * w
        bars   = ax.bar(x + offset, vals, width=w,
                        label=SCALE_LABELS[key], color=SCALE_COLORS[key])
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2)
    ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=11)
    ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE per Object Category — DA3METRIC-LARGE + Vehicle-Anchored", fontsize=12)
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save_fig(fig, "10_rmse_by_category.png")


# ── Plot 11: Scale Factor vs Depth ───────────────────────────────────────────
def plot_scale_vs_depth(df: pd.DataFrame) -> None:
    spec = [
        ("median_scale", "Median Scale (oracle)", SCALE_COLORS["median"]),
        ("mean_scale",   "Mean Scale (oracle)",   SCALE_COLORS["mean"]),
        ("affine_scale", "Affine Scale (oracle)", SCALE_COLORS["affine"]),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=col,
                    scatter_kws={"s": 18, "alpha": 0.30, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="Scale = 1.0")
        if "va_global_scale" in df:
            va_s = df["va_global_scale"].iloc[0]
            ax.axhline(va_s, color=SCALE_COLORS["va"], linestyle="-.",
                       linewidth=2.0, label=f"VA Scale = {va_s:.3f}")
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Scale Factor")
        ax.set_title(f"{label} vs Depth"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor vs Ground-Truth Depth", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "11_scale_vs_depth.png")


# ── Plot 12: Vehicle Anchor Quality ──────────────────────────────────────────
def plot_anchor_quality(df: pd.DataFrame) -> None:
    """Show depth-anchor accuracy and sub-category breakdown for vehicle masks."""
    veh = df[df["super_cat"] == "vehicle"].dropna(
        subset=["anchor_metric_depth", "anchor_raw_depth"]
    )
    if veh.empty:
        log.warning("No vehicle rows with anchor data — skipping Plot 12.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    # (a) Anchor metric depth vs LiDAR
    ax = axes[0]
    sub_cats = veh["sub_cat"].unique()
    for sc in sub_cats:
        s = veh[veh["sub_cat"] == sc]
        ax.scatter(s["lidar_avg_depth"], s["anchor_metric_depth"],
                   s=20, alpha=0.5, label=sc)
    ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
    ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Anchor Metric Depth (m)")
    ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
    ax.set_title("Vehicle Anchor Accuracy"); ax.legend(fontsize=9)

    # (b) Anchor error histogram
    ax = axes[1]
    anchor_err = veh["anchor_metric_depth"] - veh["lidar_avg_depth"]
    ax.hist(anchor_err.dropna(), bins=60, color=SCALE_COLORS["va"],
            alpha=0.75, edgecolor="white")
    ax.axvline(anchor_err.mean(), color="black", linestyle="--",
               label=f"Mean error = {anchor_err.mean():.2f} m")
    ax.set_xlabel("Anchor Error (m)"); ax.set_ylabel("Count")
    ax.set_title("Vehicle Anchor Depth Error Distribution"); ax.legend()

    # (c) d_from_height vs d_from_width scatter
    ax = axes[2]
    both = veh.dropna(subset=["d_from_height", "d_from_width"])
    if not both.empty:
        ax.scatter(both["d_from_height"], both["d_from_width"],
                   s=18, alpha=0.35, color=SCALE_COLORS["va"])
        lim_max = max(both[["d_from_height","d_from_width"]].max().max(), 1)
        ax.plot([0, lim_max], [0, lim_max], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_xlabel("Depth from Height Anchor (m)")
        ax.set_ylabel("Depth from Width Anchor (m)")
        ax.set_title("Height-Anchor vs Width-Anchor Consistency"); ax.legend()
    else:
        ax.set_visible(False)

    fig.suptitle("Vehicle Anchor Quality Analysis", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "12_vehicle_anchor_quality.png")


# ── Plot 00: Summary Table ────────────────────────────────────────────────────
def plot_summary_table(df: pd.DataFrame) -> None:
    va_s = df["va_global_scale"].mean() if "va_global_scale" in df else float("nan")
    va_t = df["va_global_shift"].mean() if "va_global_shift" in df else float("nan")
    rows_data, specs = [], [
        ("Unscaled",               "unscaled_rmse","unscaled_abs_rel","unscaled_delta1",None,None,False),
        ("Median Scale (oracle)",  "median_rmse",  "median_abs_rel",  "median_delta1",  "median_scale",None,False),
        ("Mean Scale (oracle)",    "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_scale",  None,False),
        ("Affine Scale (oracle)",  "affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_scale","affine_shift",False),
        (SCALE_LABELS["va"],"va_rmse",      "va_abs_rel",      "va_delta1",      None,None,True),
    ]
    for label, rc, ac, dc, sc, shc, is_va in specs:
        sv  = df[sc].mean()  if sc   else (va_s if is_va else 1.0)
        shv = df[shc].mean() if shc  else (va_t if is_va else 0.0)
        rows_data.append([
            label,
            f"{sv:.4f}"  if not is_va else f"{va_s:.4f}",
            f"{shv:.4f}" if (shc or is_va) else "—",
            f"{df[rc].mean():.4f}",
            f"{df[ac].mean():.4f}",
            f"{df[dc].mean():.4f}",
        ])
    fig, ax = plt.subplots(figsize=(16, 3)); ax.axis("off")
    tbl = ax.table(
        cellText=rows_data,
        colLabels=["Scale Method", "Scale", "Shift (m)",
                   "Avg RMSE (m)", "Avg Abs Rel", "Avg δ<1.25"],
        cellLoc="center", loc="center", bbox=[0, 0, 1, 1],
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
    for j in range(6):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows_data) + 1):
        bg = "#fff7e6" if i == len(rows_data) else ("white" if i % 2 == 0 else "#f0f4f8")
        for j in range(6):
            tbl[i, j].set_facecolor(bg)
    # Highlight VA row in orange
    for j in range(6):
        tbl[len(rows_data), j].set_edgecolor(SCALE_COLORS["va"])
        tbl[len(rows_data), j].set_linewidth(2)
    fig.suptitle("DA3METRIC-LARGE — Scale Method Summary (Vehicle-Anchored vs Oracle)",
                 fontsize=13, fontweight="bold", y=1.05)
    plt.tight_layout(); _save_fig(fig, "00_summary_table.png", dpi=180)


def generate_all_plots(df: pd.DataFrame) -> None:
    """Generate and save all 13 analysis plots."""
    if df.empty:
        log.warning("DataFrame is empty — skipping plot generation.")
        return
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    log.info("Generating analysis plots → %s", PLOT_DIR)
    plot_summary_table(df)
    plot_gt_vs_pred(df);           plot_gt_vs_rmse(df)
    plot_angle_vs_rmse(df);        plot_scale_comparison_bars(df)
    plot_model_vs_lidar_by_category(df)
    plot_rmse_distributions(df);   plot_scale_distributions(df)
    plot_3d_error_surfaces(df);    plot_depth_bin_analysis(df)
    plot_rmse_by_category(df);     plot_scale_vs_depth(df)
    plot_anchor_quality(df)
    log.info("All 13 plots saved to %s", PLOT_DIR)

# ══════════════════════════════════════════════════════════════════════════════
# 14.  TERMINAL SUMMARY PRINTER
# ══════════════════════════════════════════════════════════════════════════════

def print_summary(df: pd.DataFrame, ss_result: Optional[ScaleShiftResult] = None) -> None:
    W = 115
    va_s = df["va_global_scale"].iloc[0] if "va_global_scale" in df else float("nan")
    va_t = df["va_global_shift"].iloc[0] if "va_global_shift" in df else float("nan")
    print("\n" + "═" * W)
    print("  DA3METRIC-LARGE — Vehicle-Anchored Scale/Shift Correction — Summary")
    print(f"  Masks evaluated: {len(df):,}   |   Frames: {df['img_id'].nunique():,}")
    if ss_result:
        print(f"  Global scale = {ss_result.scale:.4f}  shift = {ss_result.shift:.4f}  "
              f"anchors = {ss_result.n_inliers}/{ss_result.n_anchors} inliers  "
              f"RMSE_anchor = {ss_result.rmse_m:.3f} m  method = {ss_result.method}")
        print(f"  Correction model: {'SCALE ONLY (shift fixed at 0.00)' if ss_result.only_scale else 'scale + shift'}")
    print("═" * W)
    hdr = (f"{'Scale Method':<38}"
           f"{'Scale':>10}{'Shift(m)':>10}"
           f"{'RMSE(m)':>10}{'AbsRel':>10}"
           f"{'δ<1.25':>10}{'δ<1.25²':>10}{'δ<1.25³':>10}")
    print(hdr); print("─" * W)
    specs = [
        ("Unscaled",               "unscaled_rmse","unscaled_abs_rel","unscaled_delta1","unscaled_delta2","unscaled_delta3",None,None,False),
        ("Median Scale (oracle)",  "median_rmse",  "median_abs_rel",  "median_delta1",  "median_delta2",  "median_delta3",  "median_scale",None,False),
        ("Mean Scale (oracle)",    "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_delta2",    "mean_delta3",    "mean_scale",  None,False),
        ("Affine Scale (oracle)",  "affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_delta2",  "affine_delta3",  "affine_scale","affine_shift",False),
        (SCALE_LABELS["va"],"va_rmse",      "va_abs_rel",      "va_delta1",      "va_delta2",      "va_delta3",      None,None,True),
    ]
    for row_spec in specs:
        label, rc, ac, d1c, d2c, d3c, sc, shc, is_va = row_spec
        sv  = df[sc].mean()  if sc   else (va_s if is_va else 1.0)
        shv = df[shc].mean() if shc  else (va_t if is_va else 0.0)
        marker = " ★" if is_va else ""
        print(
            f"  {label:<36}{marker}"
            f"{sv:>10.4f}{shv:>10.4f}"
            f"{df[rc].mean():>10.4f}{df[ac].mean():>10.4f}"
            f"{df[d1c].mean():>10.4f}{df[d2c].mean():>10.4f}{df[d3c].mean():>10.4f}"
        )
    print("═" * W + "\n")
    print("  ★ Vehicle-Anchored: no LiDAR ground truth used — purely from vehicle dimensions + DA3 intrinsics\n")

print('All 13 plot functions defined.')


## Cell 17 · Run Pass 1 — Anchor Collection + Global Correction

* `SKIP_PASS1 = False` → collect anchors (DA3 inference over all frames with usable vehicle masks), fit, cache.
* `SKIP_PASS1 = True`  → load the cached fit. If the cache was made with a different `ONLY_SCALE` or focal
  policy (or is an old broken `no_fit` file) it is **re-fitted from `vehicle_anchors.csv`** automatically,
  so you can flip `ONLY_SCALE` and re-run just this cell + Pass 2.
* Fewer than `MIN_ANCHORS_FOR_FIT` anchors → the cell **stops with an error** instead of continuing with an identity fit.


In [ ]:
ss_result: Optional[ScaleShiftResult] = None

# ── Index dataset ──────────────────────────────────────────────────────────────
log.info("Indexing KITTI Eigen-Split …")
val_list = index_kitti(use_train=USE_TRAIN)
assert not val_list.empty, "No frames found — check KITTI_ROOT path."
print(f"Dataset: {len(val_list)} frames, {val_list['drive'].nunique()} drives.")

# ── Load DA3 depth model ───────────────────────────────────────────────────────
da3_mdl = load_depth_model()

if not SKIP_PASS1:
    log.info("Starting Pass 1 …")
    all_anchors = run_pass1_collect_anchors(val_list, da3_mdl)
    print(f"\nTotal vehicle anchors collected: {len(all_anchors)}")

    log.info("Estimating global correction (ONLY_SCALE=%s) …", ONLY_SCALE)
    ss_result = estimate_global_scale_shift(all_anchors)      # raises if too few anchors
    report_anchor_sanity(all_anchors, ss_result)
    save_scale_cache(ss_result)
else:
    cache = load_scale_cache()
    if cache_matches_config(cache):
        ss_result = scale_result_from_cache(cache)
        log.info("Loaded cached correction: scale=%.4f  shift=%.4f  (only_scale=%s)",
                 ss_result.scale, ss_result.shift, ss_result.only_scale)
    else:
        log.warning("Cached correction does not match the current config (ONLY_SCALE / focal policy) "
                    "or is an invalid 'no_fit' file → re-fitting from %s", ANCHORS_CSV)
        anchors_from_csv = load_anchors_from_csv()
        ss_result = estimate_global_scale_shift(anchors_from_csv)
        report_anchor_sanity(anchors_from_csv, ss_result)
        save_scale_cache(ss_result)

print(f"\n{'─'*60}")
print(f"Vehicle-Anchored correction  [{'SCALE ONLY' if ss_result.only_scale else 'scale + shift'}]:")
print(f"  scale = {ss_result.scale:.4f}")
print(f"  shift = {ss_result.shift:.4f} m")
print(f"  inliers = {ss_result.n_inliers} / {ss_result.n_anchors}")
print(f"  RMSE (anchors) = {ss_result.rmse_m:.3f} m")
print(f"  method = {ss_result.method}")
print(f"{'─'*60}")

va_scale = ss_result.scale
va_shift = ss_result.shift


## Cell 18 · Run Pass 2 — Depth Evaluation

Set `SKIP_INFERENCE = True` in the config cell to skip inference and jump
directly to analysis using an existing CSV.

In [ ]:
if not SKIP_INFERENCE:
    if not os.path.exists(OUTPUT_CSV_PATH):
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(OUTPUT_CSV_PATH, index=False)
    log.info("Starting Pass 2 …")
    run_pass2_evaluate(val_list, da3_mdl, va_scale, va_shift)
else:
    log.info("Skipping Pass 2 — using existing CSV at %s", OUTPUT_CSV_PATH)

print("Pass 2 complete.")

## Cell 19 · Load Results & Print Summary

In [ ]:
assert os.path.exists(OUTPUT_CSV_PATH), f"Results CSV not found: {OUTPUT_CSV_PATH}"
df = load_results()
print(f"Loaded {len(df):,} rows from {OUTPUT_CSV_PATH}")
print_summary(df, ss_result if 'ss_result' in dir() else None)

## Cell 20 · Generate All 13 Analysis Plots

In [ ]:
if not SKIP_PLOTS:
    generate_all_plots(df)
    print(f"\nAll plots saved to: {PLOT_DIR}")
    import glob as _glob
    pngs = sorted(_glob.glob(os.path.join(PLOT_DIR, "*.png")))
    print(f"Generated {len(pngs)} plot files:")
    for p in pngs:
        print(f"  {os.path.basename(p)}")
else:
    print("SKIP_PLOTS=True — skipping plot generation.")